## Структура датасета CISI

---

### Файл 1: `CISI.ALL` — Документы

**1460 научных статей и отчётов** по библиотечному делу и информатике.

| Поле | Название | Описание |
|:----:|----------|----------|
| `.I` | ID | Уникальный номер документа |
| `.T` | Заголовок | Название статьи |
| `.A` | Автор | Автор(ы) |
| `.B` | Год | Год издания, встречается редко |
| `.W` | Текст | Основное содержание (абстракт) |
| `.X` | Ссылки | Кто и сколько раз сослался на этот документ |

 **Формат `.X`:** `ID_источника количество_ссылок ID_цели`
 - Первый столбец — кто ссылается
 - Второй столбец — сколько раз
 - Третий столбец — на кого ссылаются (всегда равен `.I` текущего документа)

---

### Файл 2: `CISI.QRY` — Запросы

**112 поисковых запросов** на естественном языке.

| Поле | Название | Описание |
|:----:|----------|----------|
| `.I` | ID | Уникальный номер запроса |
| `.W` | Текст | Текст запроса, может содержать несколько предложений |

 Может содержать и другие поля (`.T`, `.A`, `.B`, `.W`), как в `CISI.ALL`.

---

### Файл 3: `CISI.REL` — Эталонная разметка

Связи **«запрос → документ»**, определённые экспертами вручную.

| Столбец | Содержит | Примечание |
|:-------:|----------|------------|
| 1 | ID запроса | Какому запросу из `CISI.QRY` соответствует связь |
| 2 | ID документа | Какой документ из `CISI.ALL` соответствует запросу |

 **Смысл:** каждая строка говорит «запрос X связан с документом Y».  
 Используется для оценки качества поиска.

---


In [ ]:
from pathlib import Path

DATA_DIR = Path("notebook") if Path("notebook/CISI.ALL").exists() else Path(".")

def data_path(filename):
    return DATA_DIR / filename

def parse(filename):
    parse_doc = {}
    cur_doc = None
    cur_field = None

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line.startswith(".I "):
                doc_id = int(line.split()[-1])
                cur_doc = {
                    "id": doc_id,
                    "title": "",
                    "author": [],
                    "year": "",
                    "text": "",
                    "x_ref": []
                }

                if filename == "CISI.ALL": 
                    cur_doc["x_ref"] = []

                parse_doc[doc_id] = cur_doc
                cur_field = None
                
            elif line.startswith(".T"):
                cur_field = "title"
            elif line.startswith(".A"):
                cur_field = "author"
            elif line.startswith(".B"):
                cur_field = "year"
            elif line.startswith(".W"):
                cur_field = "text"
            elif line.startswith(".X"):
                cur_field = "x_ref"
            
            elif (cur_field is not None) and (cur_doc is not None):
                if cur_field == "x_ref":
                    parts = line.split()
                    cur_doc["x_ref"].append({
                        "source": int(parts[0]),
                        "count":  int(parts[1]),
                        "target": int(parts[2])
                    })
                elif cur_field == "author":
                    cur_doc["author"].append(line)
                else:
                    if cur_doc[cur_field]:
                        cur_doc[cur_field] += (" " + line)
                    else:
                        cur_doc[cur_field] = line
    return parse_doc


In [ ]:
documents = parse(data_path("CISI.ALL"))
queries = parse(data_path("CISI.QRY"))
print(documents[1])
print(queries[1])

In [ ]:
relevance = {}

with open(data_path("CISI.REL"), "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip().split()
        if not line:
            continue
        
        query_id = int(line[0])
        doc_id = int(line[1])

        if query_id not in relevance:
            relevance[query_id] = []

        relevance[query_id].append(doc_id)

print(relevance[1])

In [ ]:
print(f"Документов: {len(documents)}")
print(f"Запросов:   {len(queries)}")
print(f"Тип запроса №1: {type(queries[1])}")
print(f"Запрос №1: {queries[1]}")

## Оценка качества поисковой системы с помощью `CISI.REL`
`CISI.REL` — это **золотой стандарт** (ground truth), с которым сравниваются результаты поиска.

---

### Как это работает
Для каждого запроса:
1. Твой алгоритм выдаёт ранжированный список документов
2. `CISI.REL` говорит, какие документы должны быть найдены
3. Сравниваем и считаем метрики

---

### Основные метрики

| Метрика | Что измеряет | Формула |
|---------|-------------|:-------:|
| **Precision@K** | Доля релевантных среди первых K | `найдено верных / общее число найденных` |
| **Recall@K** | Доля найденных от всех релевантных | `найдено верных / всего релевантных из CISI.REL` |
| **AP** | Средняя точность по всем позициям | `∑Precision(i) / всего релевантных` |
| **MAP** | Средний AP по всем 112 запросам | `∑AP / 112` |

---



In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

def ensure_nltk_resource(resource_path, package_name, required=True):
    try:
        nltk.data.find(resource_path)
    except LookupError:
        downloaded = nltk.download(package_name, quiet=True)
        if not downloaded and required:
            raise RuntimeError(
                f"NLTK resource '{package_name}' is missing. "
                "Connect to the internet and rerun this cell, or install it manually."
            )
        return downloaded
    return True

ensure_nltk_resource("tokenizers/punkt", "punkt")
ensure_nltk_resource("tokenizers/punkt_tab", "punkt_tab")
ensure_nltk_resource("corpora/stopwords", "stopwords")
wordnet_available = ensure_nltk_resource("corpora/wordnet", "wordnet", required=False)

class IdentityLemmatizer:
    def lemmatize(self, word):
        return word

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer() if wordnet_available else IdentityLemmatizer()

if not wordnet_available:
    print("WordNet is unavailable; lemmatization will keep tokens unchanged.")

In [ ]:
def clean_text(text):
    word_tokens = word_tokenize(text.lower())
    filtered = [word for word in word_tokens 
                if word not in stop_words and (word.isalpha() or word.isdigit())]
    return filtered

for doc_id, doc in documents.items():
    full_text = doc["title"] + " " + doc["text"]

    if doc["author"]:
        authors_text = " ".join(doc["author"])
        full_text = full_text + " " + authors_text

    # Заголовок
    title_tokens = clean_text(doc["title"])
    doc["title_tokens"] = title_tokens
    doc["title_stemmed"] = [stemmer.stem(w) for w in title_tokens]
    doc["title_lemmatized"] = [lemmatizer.lemmatize(w) for w in title_tokens]   

    # Авторы
    author_text = " ".join(doc["author"]) if doc["author"] else ""
    author_tokens = clean_text(author_text)
    doc["author_tokens"] = author_tokens
    doc["author_stemmed"] = [stemmer.stem(w) for w in author_tokens]
    doc["author_lemmatized"] = [lemmatizer.lemmatize(w) for w in author_tokens]
    
    # Основной текст
    text_tokens = clean_text(doc["text"])
    doc["text_tokens"] = text_tokens
    doc["text_stemmed"] = [stemmer.stem(w) for w in text_tokens]
    doc["text_lemmatized"] = [lemmatizer.lemmatize(w) for w in text_tokens]

    # Весь текст сразу
    tokens = clean_text(full_text)
    doc["tokens"] = tokens
    doc["stemmed_words"] = [stemmer.stem(word) for word in tokens]
    doc["lemmatized_words"] = [lemmatizer.lemmatize(word) for word in tokens]


for query_id, query in queries.items():
    full_text = query["title"] + " " + query["text"]

    if query["author"]:
        authors_text = " ".join(query["author"])
        full_text = full_text + " " + authors_text

    # Заголовок
    title_tokens = clean_text(query["title"])
    query["title_tokens"] = title_tokens
    query["title_stemmed"] = [stemmer.stem(w) for w in title_tokens]
    query["title_lemmatized"] = [lemmatizer.lemmatize(w) for w in title_tokens]
    
    # Авторы
    author_text = " ".join(query["author"]) if query["author"] else ""
    author_tokens = clean_text(author_text)
    query["author_tokens"] = author_tokens
    query["author_stemmed"] = [stemmer.stem(w) for w in author_tokens]
    query["author_lemmatized"] = [lemmatizer.lemmatize(w) for w in author_tokens]
    
    # Основной текст
    text_tokens = clean_text(query["text"])
    query["text_tokens"] = text_tokens
    query["text_stemmed"] = [stemmer.stem(w) for w in text_tokens]
    query["text_lemmatized"] = [lemmatizer.lemmatize(w) for w in text_tokens]

    # Весь текст сразу
    tokens = clean_text(full_text)
    query["tokens"] = tokens
    query["stemmed_words"] = [stemmer.stem(word) for word in tokens]
    query["lemmatized_words"] = [lemmatizer.lemmatize(word) for word in tokens]


In [ ]:
import pandas as pd

raw_vocab = set()
cleaned_vocab = set()
stemmed_vocab = set()
lemmatized_vocab = set()

for doc_id, doc in documents.items():
    raw_text = (doc["title"] + " " + doc["text"] + " " + " ".join(doc["author"])).lower()
    raw_tokens = word_tokenize(raw_text.lower())
    raw_vocab.update(raw_tokens)
    
    cleaned_vocab.update(doc["tokens"])
    stemmed_vocab.update(doc["stemmed_words"])
    lemmatized_vocab.update(doc["lemmatized_words"])

data = {
    "Метод обработки CISI.ALL": [
        "Сырой текст (токены)",
        "После очистки (без стоп-слов и пунктуации)",
        "После стемминга (Porter Stemmer)",
        "После лемматизации (WordNet Lemmatizer)"
    ],
    "Уникальных токенов": [
        len(raw_vocab),
        len(cleaned_vocab),
        len(stemmed_vocab),
        len(lemmatized_vocab)
    ],
    "% от сырого текста": [
        100.0,
        round(len(cleaned_vocab) / len(raw_vocab) * 100, 1),
        round(len(stemmed_vocab) / len(raw_vocab) * 100, 1),
        round(len(lemmatized_vocab) / len(raw_vocab) * 100, 1)
    ],
    "Удалено токенов": [
        0,
        len(raw_vocab) - len(cleaned_vocab),
        len(raw_vocab) - len(stemmed_vocab),
        len(raw_vocab) - len(lemmatized_vocab)
    ]
}

df = pd.DataFrame(data)
df.head()

In [ ]:
raw_vocab = set()
cleaned_vocab = set()
stemmed_vocab = set()
lemmatized_vocab = set()

for query_id, query in queries.items():
    raw_text = (query["title"] + " " + query["text"] + " " + " ".join(query["author"])).lower()
    raw_tokens = word_tokenize(raw_text.lower())
    raw_vocab.update(raw_tokens)
    
    cleaned_vocab.update(query["tokens"])
    stemmed_vocab.update(query["stemmed_words"])
    lemmatized_vocab.update(query["lemmatized_words"])

data = {
    "Метод обработки CISI.QRY": [
        "Сырой текст",
        "После очистки",
        "После стемминга",
        "После лемматизации"
    ],
    "Уникальных токенов": [
        len(raw_vocab),
        len(cleaned_vocab),
        len(stemmed_vocab),
        len(lemmatized_vocab)
    ],
    "% от сырого текста": [
        100.0,
        round(len(cleaned_vocab) / len(raw_vocab) * 100, 1),
        round(len(stemmed_vocab) / len(raw_vocab) * 100, 1),
        round(len(lemmatized_vocab) / len(raw_vocab) * 100, 1)
    ],
    "Удалено токенов": [
        0,
        len(raw_vocab) - len(cleaned_vocab),
        len(raw_vocab) - len(stemmed_vocab),
        len(raw_vocab) - len(lemmatized_vocab)
    ]
}

df = pd.DataFrame(data)
df.head()

## Сравнительный анализ

**1. Очистка удаляет почти половину словаря документов**

Стоп-слова и пунктуация занимают **14.8%** в `CISI.ALL`, но только **0.9%** в `CISI.QRY`. Запросы изначально более осмысленные и содержат меньше «мусора».

**2. Стемминг сжимает сильнее лемматизации**

- `CISI.ALL`: стемминг **55.0%** vs лемматизация **75.9%** (разница 20.9%)
- `CISI.QRY`: стемминг **68.2%** vs лемматизация **88.3%** (разница 20.1%)

Стемминг агрессивнее объединяет однокоренные слова, лемматизация точнее сохраняет смысл.

**3. Запросы и документы**

Запросы короче и плотнее по смыслу — очистка удаляет меньше токенов, чем из документов.

## Построение обратного индекса

In [ ]:
from collections import defaultdict
import time
import math


def build_simple_inverted_index(documents, field):
    inverted_index = defaultdict(set)

    start = time.time()
    for doc_id, doc in documents.items():
        for word in doc[field]:
            inverted_index[word].add(doc_id)

    duration = time.time() - start
    print(f"Простой обратный индекс для поля ({field}) : {duration:.4f} сек, {len(inverted_index)} слов")
    return {word: sorted(docs) for word, docs in inverted_index.items()}
    # return inverted_index
  


def build_rich_inverted_index(documents, field):
    N = len(documents)
    inverted_index = defaultdict(lambda: defaultdict(lambda: {'tf': 0}))
    
    start = time.time()
    for doc_id, doc in documents.items():
        for word in doc[field]:
            inverted_index[word][doc_id]['tf'] += 1

    duration = time.time() - start
    
    df = {}
    idf = {}
    for word, doc_dict in inverted_index.items():
        df[word] = len(doc_dict)
        idf[word] = math.log(N / df[word])

    doc_lengths = {doc_id: len(doc[field]) for doc_id, doc in documents.items()}
    
    print(f"Обогащённый обратный индекс для поля ({field}): {duration:.4f} сек, {len(inverted_index)} слов")
    
    return {
        'inverted_index': inverted_index,
        'df': df,
        'idf': idf,
        'field': field,
        'N': N,
        'doc_lengths': doc_lengths
    }


simple_indexes = {}
rich_indexes = {}

fields = [
    "tokens", "stemmed_words", "lemmatized_words",
    "title_tokens", "title_stemmed", "title_lemmatized",
    "author_tokens", "author_stemmed", "author_lemmatized",
    "text_tokens", "text_stemmed", "text_lemmatized"
]

for field in fields:
    simple_indexes[field] = build_simple_inverted_index(documents, field)

print("")

for field in fields:
    rich_indexes[field] = build_rich_inverted_index(documents, field)



#### Выводы 
1. Простой индекс строится быстрее, но для нашего малого количества документов это не сильно ощущается
2. Стемминг дает меньше слов в индексе и строится быстрее, чем лемматизация

## Реализация алгоритмов ранжирования

In [ ]:
def boolean_search(query, query_field, index_field, operator="OR", fallback_field=None):
    query_words = query[query_field]
    idx = simple_indexes[index_field]
    
    if not query_words:
        return []
    
    sets = [set(idx.get(word, [])) for word in query_words]
    
    if not sets:
        return []
    
    if operator == "AND":
        result = set.intersection(*sets)
    elif operator == "OR":
        result = set.union(*sets)
    else:
        return set()
    
    if not result and fallback_field:
        return boolean_search(query, "stemmed_words", fallback_field, 
                             operator)
    
    return result


In [ ]:
# Тест
query_id = 1
query = queries[query_id]

results = boolean_search(query, "stemmed_words", "stemmed_words", "OR")

print(f"Запрос №{query_id}: {query['text'][:70]}...")
print(f"Найдено документов: {len(results)}")
print(f"Первые 10: {sorted(results)[:10]}")

if query_id in relevance:
    relevant_set = set(relevance[query_id])
    print(f"\nРелевантных документов: {len(relevant_set)}")
    print(f"Пересечение: {len(results & relevant_set)}")

## Булев поиск без ранжирования

Булев поиск просто находит все документы, где встречается хотя бы одно слово из запроса — без оценки, без сортировки.

### Проблема

- Найдено 1034 из 1460 документов — почти вся коллекция
- Все 46 релевантных где-то внутри, но пользователь не знает, какие из них нужные
- Порядок выдачи просто по возрастанию ID, нет сортировки по смыслу
- Пользователю придётся листать сотни документов, чтобы найти 46 нужных

### Вывод

Без скора булев поиск даёт ответ «да/нет» — документ подходит или нет.
Но он не различает:
- частичное совпадение и полное
- редкие слова и частые
- короткий точный документ и длинный размытый

Для нормального поиска нужно ранжирование — TF-IDF или BM25.

In [ ]:
def tfidf_search(query, query_field, rich_idx, normalize=True, fallback_idx=None):
    query_words = query[query_field]
    inverted_index = rich_idx['inverted_index']
    idf = rich_idx['idf']
    doc_lengths = rich_idx['doc_lengths']
    
    if not query_words:
        return []
    
    scores = defaultdict(float)
    
    for word in query_words:
        if word not in inverted_index:
            continue
        
        word_idf = idf[word]
        
        for doc_id, meta in inverted_index[word].items():
            scores[doc_id] += meta['tf'] * word_idf
    
    if not scores and fallback_idx:
        results = tfidf_search(query, "stemmed_words", fallback_idx, normalize, fallback_idx=None)
        return [(doc_id, score * 0.5) for doc_id, score in results]   

    if not scores:
        return []

    if normalize:
        for doc_id in scores:
            scores[doc_id] /= doc_lengths[doc_id]
    
    results = [(doc_id, score) for doc_id, score in scores.items()]
    results.sort(key=lambda x: x[1], reverse=True)
    
    return results

In [ ]:
query_id = 1
query = queries[query_id]
field = "stemmed_words"
idx = rich_indexes[field]

# TF-IDF без нормализации
results_no_norm = tfidf_search(query, field, idx, normalize=False)

# TF-IDF с нормализацией
results_norm = tfidf_search(query, field, idx, normalize=True)

print(f"Запрос №{query_id}: {query['text'][:70]}...")
print(f"Слов в запросе: {len(query[field])}")

print(f"\n{'Без нормализации':-^50}")
print(f"{'Doc ID':<10} {'Score':<12}")
print("-" * 22)

for doc_id, score in results_no_norm[:10]:
    print(f"{doc_id:<10} {score:<12.6f}")

print(f"\n{'С нормализацией':-^50}")
print(f"{'Doc ID':<10} {'Score':<12}")
print("-" * 22)

for doc_id, score in results_norm[:10]:
    print(f"{doc_id:<10} {score:<12.6f}")


if query_id in relevance:
    relevant_set = set(relevance[query_id])
    top10_no_norm = set(d for d, _ in results_no_norm[:10])
    top10_norm = set(d for d, _ in results_norm[:10])
    
    print(f"\nРелевантных: {len(relevant_set)}")
    print(f"Без нормализации: Precision@(K=10) = {len(top10_no_norm & relevant_set) / 10:.4f}")
    print(f"С нормализацией:   Precision@(K=10) = {len(top10_norm & relevant_set) / 10:.4f}")

## Влияние нормализации

Без нормализации длинные документы получают высокий скор — 
в них больше слов, значит больше шансов на совпадение. 
Нормализация делит скор на длину документа, уравнивая короткие и длинные.

## Что делать, если в искомой секции ничего не нашлось?

### Два подхода

**Вариант 1: Возвращать пустой список**

Если пользователь явно указал «искать в заголовках» — значит, ему нужны только заголовки. Отсутствие результатов — честный ответ.

**Вариант 2: Fallback на общий индекс с пониженным скором**

Если в заголовках пусто — возвращаем результаты из общего индекса с пометкой «возможно, вас заинтересует».

**Будем придерживаться второго варианта**



In [ ]:
def bm25_search(query, query_field, rich_idx, k1=1.2, b=0.75, fallback_idx=None):
    query_words = query[query_field]
    inverted_index = rich_idx['inverted_index']
    idf = rich_idx['idf']
    doc_lengths = rich_idx['doc_lengths']
    N = rich_idx['N']
    
    if not query_words:
        return []
    
    avg_dl = sum(doc_lengths.values()) / N
    
    scores = defaultdict(float)
    
    for word in query_words:
        if word not in inverted_index:
            continue
        
        word_idf = idf[word]
        
        for doc_id, data in inverted_index[word].items():
            tf = data['tf']
            dl = doc_lengths[doc_id]
            
            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * dl / avg_dl)
            scores[doc_id] += word_idf * numerator / denominator
    
    if not scores and fallback_idx:
        results = bm25_search(query, "stemmed_words", fallback_idx, k1, b, fallback_idx=None)
        return [(doc_id, score * 0.5) for doc_id, score in results]
    
    if not scores:
        return []
    
    results = [(doc_id, score) for doc_id, score in scores.items()]
    results.sort(key=lambda x: x[1], reverse=True)
    
    return results

In [ ]:
def precision_at_k(results, relevant, k):
    if k == 0:
        return 0.0
    top_k = set(doc_id for doc_id, _ in results[:k])
    return len(top_k & relevant) / k


def recall_at_k(results, relevant, k):
    if not relevant:
        return 0.0
    top_k = set(doc_id for doc_id, _ in results[:k])
    return len(top_k & relevant) / len(relevant)

In [ ]:
import random

random.seed(52)
test_queries = sorted(random.sample([q for q in queries.keys() if q in relevance], 5))
ks = [1, 3, 5, 10]

fields = [
    "tokens", "stemmed_words", "lemmatized_words",
    "title_tokens", "title_stemmed", "title_lemmatized",
    "author_tokens", "author_stemmed", "author_lemmatized",
    "text_tokens", "text_stemmed", "text_lemmatized"
]

print("=" * 80)
print("TF-IDF: ТЕСТИРОВАНИЕ ПО ВСЕМ ПОЛЯМ")
print("=" * 80)

for field in fields:
    print("\n" + "-" * 80)
    print(f"Поле: {field}")
    header = f"{'Q':<6}"
    for k in ks:
        header += f" {f'P@{k}':<8} {f'R@{k}':<8}"
    print(header)
    print("-" * len(header))
    
    for query_id in test_queries:
        query = queries[query_id]
        idx = rich_indexes[field]
        results = tfidf_search(query, field, idx)
        relevant_set = set(relevance[query_id])
        
        row = f"{query_id:<6}"
        for k in ks:
            p = precision_at_k(results, relevant_set, k)
            r = recall_at_k(results, relevant_set, k)
            row += f" {p:<8.4f} {r:<8.4f}"
        print(row)


print("\n" + "=" * 80)
print("BM25: ТЕСТИРОВАНИЕ ПО ВСЕМ ПОЛЯМ")
print("=" * 80)

for field in fields:
    print("\n" + "-" * 80)
    print(f"Поле: {field}")
    header = f"{'Q':<6}"
    for k in ks:
        header += f' {f'P@{k}':<8} {f'R@{k}':<8}'
    print(header)
    print("-" * len(header))
    
    for query_id in test_queries:
        query = queries[query_id]
        idx = rich_indexes[field]
        results = bm25_search(query, field, idx)
        relevant_set = set(relevance[query_id])
        
        row = f"{query_id:<6}"
        for k in ks:
            p = precision_at_k(results, relevant_set, k)
            r = recall_at_k(results, relevant_set, k)
            row += f" {p:<8.4f} {r:<8.4f}"
        print(row)

## Итоги

### 1. Влияние препроцессинга

Стемминг критически важен для маленьких коллекций.

На запросе 82 (`libraries`):

| Метод | Результат |
|-------|-----------|
| Без обработки | `library` ≠ `libraries` → 0 совпадений |
| Лемматизация | `libraries` → `library`, но `librarianship` осталось → 0 совпадений |
| Стемминг | всё → `librari` → 4 документа найдено |

Стемминг агрессивно группирует однокоренные слова. На 1460 документах это даёт прирост Recall без заметной потери Precision.

---

### 2. Сравнение алгоритмов ранжирования

| Алгоритм | Плюсы | Минусы | Когда использовать |
|----------|-------|--------|-------------------|
| **Булев поиск** | Простой, быстрый, Recall 100% | Нет сортировки, тысячи результатов | Быстрая проверка наличия |
| **TF-IDF** | Хороший баланс, понятная логика | Длинные документы в топе | Baseline для сравнения |
| **BM25** | Лучшая точность, насыщение TF | Два параметра (k1, b) | Основной выбор |

---

### 3. BM25 vs TF-IDF (stemmed_words)

| Запрос | P@1 (TF-IDF) | P@1 (BM25) | P@5 (TF-IDF) | P@5 (BM25) |
|--------|:---:|:---:|:---:|:---:|
| 7 | 0.00 | 0.00 | 0.00 | 0.00 |
| 35 | 0.00 | 1.00 | 0.40 | 0.80 |
| 55 | 1.00 | 1.00 | 1.00 | 1.00 |
| 82 | 0.00 | 0.00 | 0.40 | 0.40 |
| 95 | 0.00 | 1.00 | 0.40 | 0.60 |

BM25 находит первый релевантный документ чаще (P@1 = 0.60 против 0.20 у TF-IDF).
На маленьких значениях K (1, 3, 5) BM25 стабильно точнее.

---

### 4. Поиск по секциям

| Секция | Результат | Причина |
|--------|-----------|--------|
| `title` | Преимущественно нули | Запросы — вопросы, а не названия статей |
| `author` | Всегда нули | В запросах нет имён авторов |
| `text` | Лучший результат | Основной источник совпадений |
| `stemmed_words` (все поля) | Лучший результат | Объединяет всё |

Поиск только по тексту (`text_stemmed`) даёт почти те же результаты, что и общий индекс (`stemmed_words`). Заголовки и авторы в CISI.QRY не участвуют — запросы их не содержат.

---

### 5. Итоговая рекомендация

| Компонент | Выбор | Почему |
|-----------|-------|--------|
| Препроцессинг | Стемминг | Максимум совпадений, P@5 = 0.40 vs 0.00 без обработки |
| Поле для поиска | stemmed_words | Объединяет заголовок + авторов + текст |
| Алгоритм | BM25 | P@1 выше в 3 раза по сравнению с TF-IDF |
| Нормализация | По длине документа | Уравнивает короткие и длинные документы |

Оптимальная конфигурация для CISI: BM25 + стемминг + общий индекс + нормализация.